In [8]:
from datetime import datetime, timedelta
from collections import deque, defaultdict, Counter
import matplotlib.pyplot as plt

from aqi_pkg.db import get_session
from aqi_pkg.filters import *
from aqi_pkg.ml.clustering import *

import polars as pl

In [9]:
def get_all_data() -> pl.DataFrame:
    filter = Filter()
    loader = DataLoader(filter)

    data = loader.get_df()
    return data

In [10]:
start_time = datetime.now()
session = get_session()
plt.style.use("tableau-colorblind10")
try:
    df = get_all_data()
finally:
    session.close()
    print("Session closed.")
end_time = datetime.now()
print(f"Execution time: {end_time - start_time}")

Session closed.
Execution time: 0:00:12.086316


In [11]:
df.head()

scrape_id,lat,lon,locationId,city,state,country,last_updated,AQI_IN,AQI_US,CO_PPB,NO2_PPB,O3_PPB,SO2_PPB,PM1_UGM3,PM2_5_UGM3,PM10_UGM3,H_PERCENT,T_C,TVOC_PPM,Noise_DB
i64,f64,f64,str,str,str,str,datetime[μs],i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
362829,24.0299,73.0463,"""VIR11283""","""Khedbrahma""","""Gujarat""","""India""",2025-11-14 18:34:00,97,92,387.0,20.0,43.0,2.0,null,31.0,97.0,28.0,23.0,null,null
18798114,28.666833,77.119268,"""2555""","""New Delhi""","""Delhi""","""India""",2026-02-25 02:10:00,328,206,853.0,26.0,1.0,24.0,null,157.0,209.0,77.0,18.0,null,null
9030653,26.1541,81.8114,"""PLLODA000554""","""Fyzabad""","""Uttar Pradesh""","""India""",2026-01-12 23:29:00,349,259,634.0,3.0,15.0,22.0,null,184.0,197.0,43.0,11.0,null,null
7392322,28.7091,76.8182,"""PLLODA000203""","""Rohtak""","""Haryana""","""India""",2026-01-04 14:11:00,280,192,846.0,16.0,11.0,3.0,null,114.0,171.0,63.0,15.0,null,null
6518437,27.9977,73.3486,"""PLLODA000447""","""Bikaner""","""Rajasthan""","""India""",2025-12-30 20:29:00,210,177,97.0,53.0,107.0,6.0,null,93.0,182.0,18.0,20.0,null,null


In [19]:
df.group_by("country").len().sort("len", descending=True)

country,len
str,u32
"""India""",15667005
"""Nepal""",470721
"""Pakistan""",40197
"""Thailand""",999
"""China""",264
…,…
"""Bangladesh""",12
"""Turkmenistan""",6
"""Burma""",3


# Investigating 3

In [12]:
counts = df.group_by("locationId").len()
print(counts.describe())

shape: (9, 3)
┌────────────┬────────────┬─────────────┐
│ statistic  ┆ locationId ┆ len         │
│ ---        ┆ ---        ┆ ---         │
│ str        ┆ str        ┆ f64         │
╞════════════╪════════════╪═════════════╡
│ count      ┆ 3492       ┆ 3492.0      │
│ null_count ┆ 0          ┆ 0.0         │
│ mean       ┆ null       ┆ 4633.408076 │
│ std        ┆ null       ┆ 5352.182666 │
│ min        ┆ -100165    ┆ 1.0         │
│ 25%        ┆ null       ┆ 3.0         │
│ 50%        ┆ null       ┆ 2398.0      │
│ 75%        ┆ null       ┆ 8487.0      │
│ max        ┆ VIR9985    ┆ 13380.0     │
└────────────┴────────────┴─────────────┘


In [13]:
df_three = counts.filter(pl.col("len") == 3)

In [17]:
df.filter(pl.col("locationId").is_in(df_three["locationId"].implode())).group_by("country").len().sort("len", descending=True)

country,len
str,u32
"""India""",1839
"""Thailand""",999
"""China""",264
"""Kyrgyzstan""",219
"""Iran""",120
…,…
"""Nepal""",9
"""Turkmenistan""",6
"""Myanmar""",3


In [ ]:
df_threesremoved = df.filter(~pl.col("locationId").is_in(df_three["locationId"].implode()))
df_threesremoved.group_by("country").len().sort("len", descending=True)